# SokoRadar: building our forecasting model from scratch

This is the main modelling notebook for SokoRadar. I'm building our own model step by step, and the existing prototype in the repo (the `sokoradar` model) is my benchmark, M0. If I beat it, we've improved the project. If I don't, I'll be able to say exactly why.

What happens in this notebook, in order:

1. Load the county price table from the data pipeline and check it's the right version
2. Look at how hard the simplest forecast ("next month = this month") is to beat
3. Build the targets and the train/test split
4. Build one scoring harness that marks every model the same way
5. Baselines, then the model ladder M1 to M5, one change per version
6. The comparison table and the decision log

Owner: Franklin (machine-learning side and model evaluation). Reviewer: Daniella.

## 1. Setup

In [ ]:
# Import the libraries this notebook uses
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

The settings below are the choices I might want to change later. Keeping them in one cell means I change them once and re-run, rather than hunting through the notebook.

`TARGET_SET` is the big one. On the 30 Oct call we agreed to train on all county prices, estimated ones included, until the group decides otherwise. So it's set to `"all"`. If the group switches to observed prices only, I change it to `"observed"` and re-run everything.

`TEST_YEAR` is set to 2024 for now, not 2026. We agreed to train on 2006 to 2025 and test on 2026, but the pipeline's county table stops at December 2024, so 2025 and 2026 aren't in it yet. Until the pipeline is extended, 2024 is the latest year I can hold back as a test.

In [ ]:
# Settings I can change in one place
TARGET_SET = "all"          # "all" = every county price (agreed for now) | "observed" = market-recorded prices only
TEST_YEAR = 2024            # held back until the final model is chosen; becomes 2026 once the pipeline covers it
HORIZONS = [1, 2, 3]        # forecast 1, 2 and 3 months ahead
DATA_FILE = "SokoRadar_County_2006_2024_long.csv"
EXPECTED_ROWS = 278_616     # rows in the pipeline output we checked on 29 Oct

In [ ]:
# Find the pipeline's output folder, wherever the repo was cloned
candidates = []
if os.environ.get("SOKORADAR_OUTPUT"):
    candidates.append(Path(os.environ["SOKORADAR_OUTPUT"]))
here = Path.cwd()
for up in [here, *here.parents][:5]:
    candidates.append(up / "output")

DATA_PATH = next((c / DATA_FILE for c in candidates if (c / DATA_FILE).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Run sokoradar_pipeline.py first, or set the "
        "SOKORADAR_OUTPUT environment variable to the folder that holds it."
    )
print("Using:", DATA_PATH)

## 2. Decision log

Every real choice in this notebook goes into the decision log: what I chose, what I rejected, and why. It's printed as a table at the end, and it's how anyone (including future me) can see why the model looks the way it does.

In [ ]:
# Decision log helper: records each choice, the evidence and the options I rejected
DECISIONS = []

def log_decision(step, question, evidence, decision, rejected=""):
    DECISIONS.append({
        "step": step,
        "question": question,
        "evidence": evidence,
        "decision": decision,
        "rejected": rejected,
    })

In [ ]:
# Log the decisions Daniella and I agreed on the 30 Oct call
log_decision(
    "Approach",
    "Rebuild the model or improve the existing prototype?",
    "Graded and questioned on the modelling; the prototype arrived as one finished upload",
    "Build our own model from scratch; the existing sokoradar model is M0, the benchmark",
    "Improve the existing model in place",
)
log_decision(
    "Targets",
    "Which prices does the model learn from?",
    "Only 5.5% of the 278,616 county prices are observed in a market; the rest are estimated",
    "All county prices, estimated ones included (provisional, until the group decides)",
    "Observed prices only",
)
log_decision(
    "Evaluation",
    "Which prices is accuracy reported on?",
    "Estimated prices are smoother than real ones, so scores on them look better than reality",
    "Always report accuracy on observed prices as well as on all prices",
    "Report on all prices only",
)
log_decision(
    "Split",
    "Which period is held back as the test?",
    "Agreed: train 2006-2025, test 2026. The county table stops at Dec 2024",
    f"Test on {TEST_YEAR} for now; move to 2026 once the pipeline covers it",
    "Wait for the pipeline before starting",
)

## 3. Load the data

In [ ]:
# Load the county price table from the pipeline output
prices = pd.read_csv(DATA_PATH, parse_dates=["month"])
prices.head()

Before I trust the file, I check it's the version we expect. If someone regenerates the pipeline and something changes, these checks stop the notebook with a clear message instead of letting me work on the wrong data without noticing.

In [ ]:
# Check this is the expected version of the file
assert len(prices) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS:,} rows, found {len(prices):,}"
assert prices["month"].min() == pd.Timestamp("2006-01-01"), "Data should start in Jan 2006"
assert prices["month"].max() == pd.Timestamp("2024-12-01"), "Data should end in Dec 2024"
for col in ["county", "commodity", "price_type", "price_kes", "flag"]:
    assert col in prices.columns, f"Missing column: {col}"
assert prices["price_kes"].notna().all(), "Some prices are blank"
print("All checks passed:", f"{len(prices):,} rows,", prices["month"].min().date(), "to", prices["month"].max().date())

In [ ]:
# Mark which prices were recorded in a market (observed) and which were estimated
prices["observed"] = prices["flag"].str.startswith("Observed")
prices["observed"].value_counts()

In [ ]:
# Share of observed vs estimated prices, overall
share = prices["observed"].map({True: "Observed", False: "Estimated"}).value_counts(normalize=True)
(share * 100).round(1).rename("percent of all prices")

In [ ]:
# Observed prices per year
per_year = prices.groupby(prices["month"].dt.year)["observed"].agg(total="size", observed="sum")
per_year["observed_pct"] = (per_year["observed"] / per_year["total"] * 100).round(1)
per_year

**My notes:** _(write what I notice after running the cells above)_

## 4. How hard is "next month = this month" to beat?

The naive forecast says next month's price will be the same as this month's. Food prices are sticky, so it's often hard to beat. Before building any model, I want to know how good it already is, because that tells me how much room the models have to improve.

I measure the naive error twice: on all prices, and on observed prices only. If the two differ a lot, that shows how much the estimated prices flatter the scores.

In [ ]:
# Naive forecast error at each horizon, on all prices and on observed prices only
key = ["county", "commodity", "price_type"]

def naive_error(df, h):
    now = df[key + ["month", "price_kes", "observed"]]
    past = now.assign(month=now["month"] + pd.DateOffset(months=h))
    pairs = now.merge(past, on=key + ["month"], suffixes=("", "_past"))
    pairs["abs_pct_err"] = (pairs["price_kes"] - pairs["price_kes_past"]).abs() / pairs["price_kes"] * 100
    return pairs

rows = []
for h in HORIZONS:
    pairs = naive_error(prices, h)
    obs_pairs = pairs[pairs["observed"] & pairs["observed_past"]]
    rows.append({
        "horizon (months)": h,
        "naive MAE % (all prices)": round(pairs["abs_pct_err"].mean(), 2),
        "naive MAE % (observed only)": round(obs_pairs["abs_pct_err"].mean(), 2),
        "observed pairs": len(obs_pairs),
    })
naive_table = pd.DataFrame(rows)
naive_table

In [ ]:
# Size of month-on-month price changes, observed prices only
one_month = naive_error(prices, 1)
one_month = one_month[one_month["observed"] & one_month["observed_past"]]
change_pct = (one_month["price_kes"] / one_month["price_kes_past"] - 1) * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(change_pct.clip(-40, 40), bins=80)
ax.set_xlabel("month-on-month price change (%), clipped at +/-40")
ax.set_ylabel("number of observed price pairs")
ax.set_title("Most months, observed prices barely move")
plt.show()
change_pct.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(2)

**My notes:** _(how good is naive already, and how much room is left for the models?)_

## 5. Targets and train/test split (Day 2)

_To build on Day 2: the target table (price 1, 2 and 3 months ahead for each county, crop and price type), the rolling-origin backtest folds over the training years, and the sealed test year._

In [ ]:
# Day 2: build the target table and the backtest folds

## 6. Scoring harness (Day 2)

_One function, `evaluate(model)`, that marks every model the same way: the same backtest months, the same answer key and the same measures. It reports every measure on all prices and on observed prices only. Daniella checks it in my Day 2 pull request._

In [ ]:
# Day 2: the scoring harness, evaluate(model)

## 7. Baselines (Day 3)

_Naive and seasonal naive, scored through the harness. These are the bars every model must clear._

In [ ]:
# Day 3: naive and seasonal-naive baselines through the harness

## 8. Model ladder (Days 3 to 5)

Each version changes one thing and is compared with the version before it:

- **M0:** the existing sokoradar model (benchmark)
- **M1:** LightGBM on price history + calendar
- **M2:** M1 + rainfall and vegetation
- **M3:** + related-crop prices, tested against a fixed base
- **M4:** + inflation, fuel and exchange rate
- **M5:** + tuned LightGBM (tuning on training months only)

In [ ]:
# Day 3: M1, LightGBM on price history + calendar

In [ ]:
# Day 3: M2, add rainfall and vegetation

In [ ]:
# Day 4: M3, add related-crop prices (fixed-base test)

In [ ]:
# Day 4: M4, add inflation, fuel and exchange rate

In [ ]:
# Day 5: M5, tuned LightGBM

## 9. Comparison table

In [ ]:
# Experiment table: every version, scored on all prices and on observed prices
EXPERIMENTS = pd.DataFrame(columns=[
    "version", "change", "horizon",
    "MAE % (all)", "MAE % (observed)", "vs naive (observed)", "MASE (observed)",
])
EXPERIMENTS

## 10. Decision log

In [ ]:
# Show every decision made in this notebook
pd.DataFrame(DECISIONS)